# Notebook 02 — Pipeline PySpark con Arquitectura Medallion

**Proyecto:** Sistema de Predicción de Demanda Hospitalaria — EsSalud
**Curso:** Big Data DD283 (2026-1) · **Grupo 3** · Junior Ortiz
**Semana 2 del cronograma** — criterio de éxito: *Bronze→Silver→Gold; join 4 fuentes; CIE-10 normalizado; Parquet Gold*

---

## Objetivo

Documentar la ejecución del pipeline ETL implementado en `src/etl_medallion.py`,
mostrando la evidencia de cada capa de la arquitectura Medallion.

## Arquitectura

| Capa | Responsabilidad | Salida |
|------|-----------------|--------|
| **BRONCE** | Ingesta de las 4 fuentes crudas, sin transformar | `data/bronze/` |
| **PLATA** | Limpieza, estandarización, CIE-10, grupos etarios, rezagos, JOIN | `data/silver/` |
| **ORO** | Tablas analíticas agregadas para ML y dashboard | `data/gold/` |

## Fuentes integradas

| Fuente | Archivo | Granularidad |
|--------|---------|--------------|
| HIS EsSalud | `atenciones_essalud.csv` | 1 fila = 1 atención |
| SENAMHI | `clima_lima_2022_2024.csv` | 1 fila = 1 semana |
| MINSA/CDC | `epidemio_minsa_2022_2024.csv` | 1 fila = 1 semana |
| Oferta EsSalud | `oferta_hospitalaria.csv` | 1 fila = 1 semana × establecimiento |

> **Nota:** este notebook es la *evidencia documentada* del pipeline.
> La versión productiva y reproducible vive en `src/etl_medallion.py`.

In [1]:
import os
import sys

# Configuracion Hadoop/winutils (requerido por Spark en Windows)
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

# Permite importar las funciones del pipeline productivo desde src/
sys.path.append(os.path.abspath(".."))

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct

from src.etl_medallion import (
    clave_semana_iso,
    RUTA_ATENCIONES, RUTA_EPIDEMIO, RUTA_CLIMA, RUTA_OFERTA,
    BRONCE_ATENCIONES, BRONCE_EPIDEMIO, BRONCE_CLIMA, BRONCE_OFERTA,
    RUTA_PLATA, PLATA_OFERTA,
    RUTA_ORO, ORO_DEMANDA_SEMANAL, ORO_CORRELACION, ORO_BRECHAS,
)

# El notebook se ejecuta desde notebooks/, pero las rutas del pipeline
# son relativas a la raiz del proyecto
os.chdir(os.path.abspath(".."))
print("Directorio de trabajo:", os.getcwd())

spark = (SparkSession.builder
         .appName("Notebook02_Medallion_Grupo3")
         .config("spark.driver.memory", "2g")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Directorio de trabajo: c:\Users\Home\Desktop\Grupo3\bigdata-g3-demanda-hospitalaria
Spark version: 3.5.1


In [2]:
# ============================================================
#  CAPA BRONCE — ingesta de las 4 fuentes crudas
# ============================================================
# Bronce guarda las fuentes EXACTAMENTE como llegan, sin corregir nada.
# Es la version auditable del origen: si mas adelante se detecta un error,
# siempre se puede volver aqui a verificar como venia el dato original.

fuentes = [
    ("HIS EsSalud (atenciones)",     RUTA_ATENCIONES, BRONCE_ATENCIONES),
    ("MINSA/CDC (epidemiologia)",    RUTA_EPIDEMIO,   BRONCE_EPIDEMIO),
    ("SENAMHI (clima)",              RUTA_CLIMA,      BRONCE_CLIMA),
    ("EsSalud (oferta hospitalaria)", RUTA_OFERTA,    BRONCE_OFERTA),
]

print("=" * 62)
print("CAPA BRONCE — 4 FUENTES INGESTADAS")
print("=" * 62)

for nombre, csv_origen, parquet_bronce in fuentes:
    df_b = spark.read.parquet(parquet_bronce)
    print(f"\n{nombre}")
    print(f"   Origen : {csv_origen}")
    print(f"   Bronce : {parquet_bronce}")
    print(f"   Filas  : {df_b.count():,}  |  Columnas: {len(df_b.columns)}")

CAPA BRONCE — 4 FUENTES INGESTADAS

HIS EsSalud (atenciones)
   Origen : data/raw/atenciones_essalud.csv
   Bronce : data/bronze/atenciones
   Filas  : 500,000  |  Columnas: 28

MINSA/CDC (epidemiologia)
   Origen : data/raw/epidemio_minsa_2022_2024.csv
   Bronce : data/bronze/epidemio
   Filas  : 157  |  Columnas: 5

SENAMHI (clima)
   Origen : data/raw/clima_lima_2022_2024.csv
   Bronce : data/bronze/clima
   Filas  : 157  |  Columnas: 6

EsSalud (oferta hospitalaria)
   Origen : data/raw/oferta_hospitalaria.csv
   Bronce : data/bronze/oferta
   Filas  : 471  |  Columnas: 9


In [3]:
# ------------------------------------------------------------
#  Esquema de la fuente principal (HIS EsSalud)
# ------------------------------------------------------------
# Las 28 columnas corresponden al esquema definido en el README (seccion 5.1).
# Se muestran tal cual llegan, sin transformar: esto es Bronce.

import pandas as pd

bronce_aten = spark.read.parquet(BRONCE_ATENCIONES)

esquema = pd.DataFrame(
    [(i + 1, campo.name, campo.dataType.simpleString())
     for i, campo in enumerate(bronce_aten.schema.fields)],
    columns=["#", "columna", "tipo"]
)

print(f"HIS EsSalud — {len(esquema)} columnas\n")
display(esquema)

HIS EsSalud — 28 columnas



,#,columna,tipo
0,1,atencion_id,string
1,2,fecha_atencion,date
2,3,semana_epidemiologica,int
3,4,establecimiento_id,string
4,5,establecimiento_nombre,string
5,6,red_asistencial,string
6,7,tipo_atencion,string
7,8,especialidad,string
8,9,medico_id,string
9,10,paciente_edad,int


---

## Hallazgo de calidad de datos: clave de semana epidemiológica

Durante la construcción de la capa Plata se detectó que **tres de las cuatro fuentes**
(atenciones, epidemiología y oferta) traían la clave de semana mal formada.

### El problema

La clave `AAAA-Www` se construía concatenando el **año del calendario** con el número
de semana. Según la norma **ISO 8601** (usada por MINSA/CDC para las semanas
epidemiológicas), los primeros días de enero pueden pertenecer a la semana 52 del año
anterior:

| Fecha | Semana epi. | Clave generada | Clave correcta |
|-------|-------------|----------------|----------------|
| 2022-01-01 | 52 | `2022-W52` ❌ | `2021-W52` |
| 2022-12-28 | 52 | `2022-W52` | `2022-W52` ✅ |
| 2023-01-01 | 52 | `2023-W52` ❌ | `2022-W52` |
| 2024-12-30 | 1  | `2024-W01` ❌ | `2025-W01` |

### Por qué era crítico

Dos filas distintas competían por la clave `2022-W52`. Como el JOIN resolvía la
colisión con `dropDuplicates()`, **Spark elegía una de las dos de forma arbitraria**:
el pipeline no era determinista y podía producir resultados distintos en cada ejecución.

**Impacto medido:** 2,252 atenciones con clave incorrecta y 3,613 registros con el
valor de dengue equivocado.

### La corrección

Se implementó `clave_semana_iso()` en la capa Plata (no en Bronce, que conserva el
dato original auditable, ni en el generador, para no alterar los datos ya entregados
en la EP). La regla:

- Mes **enero** y semana **≥ 52** → año **anterior**
- Mes **diciembre** y semana **= 1** → año **siguiente**
- En otro caso → año del calendario

Adicionalmente se reemplazó `dropDuplicates()` por una **validación que detiene el
pipeline** si vuelven a aparecer claves duplicadas (principio de *Veracidad*).

In [4]:
# ============================================================
#  EVIDENCIA DEL BUG — se reproduce sobre los datos de BRONCE
# ============================================================
# Bronce conserva el dato tal como llego, asi que aqui todavia
# se puede observar la clave mal formada.

from pyspark.sql.functions import month, year, when, to_date, weekofyear

epi_bronce = spark.read.parquet(BRONCE_EPIDEMIO)

print("MINSA/CDC — clave original vs. clave ISO 8601")
print(f"   Filas totales        : {epi_bronce.count()}")
print(f"   Claves distintas ANTES: {epi_bronce.select('semana').distinct().count()}  <-- hay colision")

# Se aplica la funcion del pipeline productivo
epi_corregido = (epi_bronce
    .withColumn("fecha", to_date(col("fecha")))
    .withColumn("semana_iso", clave_semana_iso(col("fecha"), weekofyear(col("fecha")))))

print(f"   Claves distintas DESPUES: {epi_corregido.select('semana_iso').distinct().count()}  <-- 1 clave por semana")

print("\nFilas cuya clave cambio:")
(epi_corregido
    .filter(col("semana") != col("semana_iso"))
    .select("fecha", "semana", "semana_iso", "casos_dengue")
    .show(truncate=False))

print("Claves duplicadas que quedan tras la correccion:")
(epi_corregido.groupBy("semana_iso")
    .agg(count("*").alias("n_filas"))
    .filter(col("n_filas") > 1)
    .show(truncate=False))

MINSA/CDC — clave original vs. clave ISO 8601
   Filas totales        : 157
   Claves distintas ANTES: 156  <-- hay colision
   Claves distintas DESPUES: 157  <-- 1 clave por semana

Filas cuya clave cambio:
+----------+--------+----------+------------+
|fecha     |semana  |semana_iso|casos_dengue|
+----------+--------+----------+------------+
|2022-01-01|2022-W52|2021-W52  |254         |
+----------+--------+----------+------------+

Claves duplicadas que quedan tras la correccion:
+----------+-------+
|semana_iso|n_filas|
+----------+-------+
+----------+-------+



In [5]:
# ============================================================
#  CAPA PLATA — limpieza, estandarizacion y JOIN de fuentes
# ============================================================
# Responsabilidades segun README (seccion 4):
#   - deduplicacion + imputacion de nulos
#   - diagnosticos -> CIE-10 normalizado
#   - edad -> grupos etarios (0-5, 6-17, 18-59, 60+)
#   - clima -> variables rezagadas 7/14/21 dias
#   - JOIN de las 4 fuentes por semana epidemiologica

plata = spark.read.parquet(RUTA_PLATA)
plata_oferta = spark.read.parquet(PLATA_OFERTA)

print("=" * 62)
print("CAPA PLATA")
print("=" * 62)
print(f"Atenciones: {plata.count():,} filas x {len(plata.columns)} columnas")
print(f"Oferta    : {plata_oferta.count():,} filas x {len(plata_oferta.columns)} columnas")

# Columnas nuevas que agrego la capa Plata respecto a Bronce
nuevas = [c for c in plata.columns if c not in bronce_aten.columns]
print(f"\nColumnas agregadas por la capa Plata ({len(nuevas)}):")
for c in nuevas:
    print(f"   - {c}")

CAPA PLATA
Atenciones: 500,000 filas x 40 columnas
Oferta    : 471 filas x 9 columnas

Columnas agregadas por la capa Plata (12):
   - semana
   - cie10_capitulo
   - grupo_etario
   - casos_dengue
   - casos_influenza
   - temperatura_min
   - temp_max_lag7
   - temp_max_lag14
   - temp_max_lag21
   - precip_lag7
   - precip_lag14
   - precip_lag21


In [6]:
# ============================================================
#  VALIDACIONES DE CALIDAD DE LA CAPA PLATA
# ============================================================

print("1) DEDUPLICACION — atencion_id debe ser unico")
total_p = plata.count()
ids_unicos = plata.select("atencion_id").distinct().count()
print(f"   Filas: {total_p:,}  |  IDs unicos: {ids_unicos:,}  |  "
      f"{'OK' if total_p == ids_unicos else 'ERROR'}\n")

print("2) GRUPOS ETARIOS — README: 0-5, 6-17, 18-59, 60+")
(plata.groupBy("grupo_etario")
      .agg(count("*").alias("atenciones"))
      .orderBy("grupo_etario")
      .show(truncate=False))

print("3) CIE-10 NORMALIZADO — capitulos detectados")
(plata.groupBy("cie10_capitulo")
      .agg(count("*").alias("atenciones"),
           countDistinct("diagnostico_cie10").alias("codigos_distintos"))
      .orderBy("cie10_capitulo")
      .show(truncate=False))

print("4) INTEGRIDAD DEL JOIN — nulos por columna integrada")
cols_join = ["casos_dengue", "casos_influenza", "temperatura_min",
             "temp_max_lag7", "temp_max_lag14", "temp_max_lag21",
             "precip_lag7", "precip_lag14", "precip_lag21"]

nulos = pd.DataFrame(
    [(c, plata.filter(col(c).isNull()).count()) for c in cols_join],
    columns=["columna", "nulos"]
)
nulos["% del total"] = (nulos["nulos"] / total_p * 100).round(2)
display(nulos)

1) DEDUPLICACION — atencion_id debe ser unico
   Filas: 500,000  |  IDs unicos: 500,000  |  OK

2) GRUPOS ETARIOS — README: 0-5, 6-17, 18-59, 60+
+------------+----------+
|grupo_etario|atenciones|
+------------+----------+
|0-5         |27595     |
|18-59       |233536    |
|6-17        |66971     |
|60+         |171898    |
+------------+----------+

3) CIE-10 NORMALIZADO — capitulos detectados
+--------------+----------+-----------------+
|cie10_capitulo|atenciones|codigos_distintos|
+--------------+----------+-----------------+
|A             |87647     |1                |
|E             |88409     |1                |
|I             |87779     |1                |
|J             |162959    |2                |
|S             |73206     |1                |
+--------------+----------+-----------------+

4) INTEGRIDAD DEL JOIN — nulos por columna integrada


,columna,nulos,% del total
0,casos_dengue,876,0.18
1,casos_influenza,876,0.18
2,temperatura_min,876,0.18
3,temp_max_lag7,1776,0.36
4,temp_max_lag14,4989,1.00
5,temp_max_lag21,8178,1.64
6,precip_lag7,1776,0.36
7,precip_lag14,4989,1.00
8,precip_lag21,8178,1.64


In [7]:
# ============================================================
#  CAPA ORO — tablas analiticas para ML y dashboard
# ============================================================
# El README (seccion 4) especifica las tablas Gold del proyecto.
# 'prediccion_proximas_4_semanas' no se genera aqui: es la salida
# del modelo Prophet (Semana 6), no una agregacion del ETL.

tablas_oro = [
    ("demanda_semanal_por_especialidad",           ORO_DEMANDA_SEMANAL),
    ("correlacion_clima_enfermedad",               ORO_CORRELACION),
    ("brechas_oferta_demanda_por_establecimiento", ORO_BRECHAS),
    ("demanda_diaria (serie para Prophet)",        RUTA_ORO),
]

print("=" * 62)
print("CAPA ORO — TABLAS ANALITICAS")
print("=" * 62)

oro = {}
for nombre, ruta in tablas_oro:
    df_o = spark.read.parquet(ruta)
    oro[nombre] = df_o
    print(f"\n{nombre}")
    print(f"   Ruta   : {ruta}")
    print(f"   Filas  : {df_o.count():,}  |  Columnas: {len(df_o.columns)}")

CAPA ORO — TABLAS ANALITICAS

demanda_semanal_por_especialidad
   Ruta   : data/gold/demanda_semanal_por_especialidad
   Filas  : 624  |  Columnas: 7

correlacion_clima_enfermedad
   Ruta   : data/gold/correlacion_clima_enfermedad
   Filas  : 156  |  Columnas: 14

brechas_oferta_demanda_por_establecimiento
   Ruta   : data/gold/brechas_oferta_demanda_por_establecimiento
   Filas  : 468  |  Columnas: 16

demanda_diaria (serie para Prophet)
   Ruta   : data/gold/demanda_diaria
   Filas  : 13,152  |  Columnas: 7


In [8]:
# ------------------------------------------------------------
#  Muestra del contenido de cada tabla Oro
# ------------------------------------------------------------

print(">>> demanda_semanal_por_especialidad")
display(oro["demanda_semanal_por_especialidad"]
        .orderBy("semana", "especialidad").limit(8).toPandas())

print(">>> correlacion_clima_enfermedad")
display(oro["correlacion_clima_enfermedad"]
        .select("semana", "total_atenciones", "temperatura_max",
                "casos_dengue", "casos_influenza", "temp_max_lag7")
        .orderBy("semana").limit(8).toPandas())

print(">>> brechas_oferta_demanda_por_establecimiento")
display(oro["brechas_oferta_demanda_por_establecimiento"]
        .select("semana", "establecimiento_nombre", "demanda_atenciones",
                "consultas_programadas", "brecha_consultas",
                "ratio_demanda_oferta")
        .orderBy("semana", "establecimiento_nombre").limit(8).toPandas())

>>> demanda_semanal_por_especialidad


,semana,especialidad,total_atenciones,dias_con_atencion,tasa_ocupacion_promedio,lista_espera_promedio,costo_total_soles
0,2022-W01,Cardiologia,563,7,0.924458,18.232682,466206.09
1,2022-W01,Emergencia,1009,7,0.927770,9.346878,808602.97
2,2022-W01,Medicina Interna,1183,7,0.923855,17.882502,943972.93
3,2022-W01,Traumatologia,458,7,0.928799,8.227074,367663.24
4,2022-W02,Cardiologia,597,7,0.932596,18.324958,465040.38
5,2022-W02,Emergencia,966,7,0.927195,9.800207,773353.37
6,2022-W02,Medicina Interna,1167,7,0.918132,18.461011,914148.10
7,2022-W02,Traumatologia,459,7,0.923660,9.283224,362000.65


>>> correlacion_clima_enfermedad


,semana,total_atenciones,temperatura_max,casos_dengue,casos_influenza,temp_max_lag7
0,2022-W01,3213,28.004451,252.0,0.0,27.8
1,2022-W02,3189,27.943838,298.0,7.0,28.4
2,2022-W03,3288,27.981326,269.0,14.0,28.9
3,2022-W04,3174,27.987587,322.0,10.0,27.6
4,2022-W05,3218,28.423275,290.0,17.0,28.4
5,2022-W06,3239,28.478574,331.0,0.0,28.1
6,2022-W07,3242,28.472301,202.0,45.0,29.8
7,2022-W08,3203,28.498751,187.0,13.0,25.5


>>> brechas_oferta_demanda_por_establecimiento


,semana,establecimiento_nombre,demanda_atenciones,consultas_programadas,brecha_consultas,ratio_demanda_oferta
0,2022-W01,Hospital Almenara,1049,736,313,1.425272
1,2022-W01,Hospital Rebagliati,1069,888,181,1.203829
2,2022-W01,Hospital Sabogal,1095,880,215,1.244318
3,2022-W02,Hospital Almenara,1067,797,270,1.338770
4,2022-W02,Hospital Rebagliati,1025,835,190,1.227545
5,2022-W02,Hospital Sabogal,1097,772,325,1.420984
6,2022-W03,Hospital Almenara,1151,909,242,1.266227
7,2022-W03,Hospital Rebagliati,1040,805,235,1.291925


In [9]:
# ------------------------------------------------------------
#  VALIDACION: coherencia entre oferta y demanda
# ------------------------------------------------------------
brechas = oro["brechas_oferta_demanda_por_establecimiento"]

total_b = brechas.count()
negativas = brechas.filter(col("brecha_consultas") < 0).count()

print(f"Filas totales            : {total_b}")
print(f"Con brecha negativa      : {negativas}  ({negativas/total_b*100:.1f}%)")
print(f"Con brecha positiva      : {total_b - negativas}\n")

display(brechas.select("demanda_atenciones", "consultas_programadas",
                       "consultas_ejecutadas", "brecha_consultas",
                       "ratio_demanda_oferta", "tasa_ocupacion_promedio")
        .summary("min", "25%", "50%", "75%", "max").toPandas())

Filas totales            : 468
Con brecha negativa      : 0  (0.0%)
Con brecha positiva      : 468



,summary,demanda_atenciones,consultas_programadas,consultas_ejecutadas,brecha_consultas,ratio_demanda_oferta,tasa_ocupacion_promedio
0,min,957,706,699,101,1.112008072653885,0.9095319961795606
1,25%,1042,792,808,161,1.1803468208092485,0.9204474885844752
2,50%,1065,852,876,213,1.2511467889908257,0.923545966228893
3,75%,1089,907,931,266,1.335811648079306,0.9262152466367717
4,max,1151,1002,1081,334,1.428754813863928,0.9379593908629443


---

## Conclusiones — Semana 2

### Criterio de éxito del README

| Requisito | Evidencia en este notebook |
|-----------|---------------------------|
| Bronze → Silver → Gold | Celdas 3, 7 y 9 |
| Join 4 fuentes | Bronce ingesta 4 fuentes; Plata las integra por semana epidemiológica |
| CIE-10 normalizado | `diagnostico_cie10` en mayúsculas + `cie10_capitulo` (celda 8) |
| Parquet Gold | 4 tablas escritas en `data/gold/` (celda 9) |

### Resultados

- **Bronce:** 4 fuentes ingestadas sin transformar — 500,000 atenciones (28 columnas), 157 semanas de clima, 157 de epidemiología, 471 registros de oferta.
- **Plata:** 500,000 filas × 40 columnas. La capa agregó 12 columnas: clave de semana ISO, capítulo CIE-10, grupo etario, casos MINSA, temperatura mínima y 6 variables rezagadas (7/14/21 días).
- **Oro:** 4 tablas analíticas — `demanda_semanal_por_especialidad` (624), `correlacion_clima_enfermedad` (156), `brechas_oferta_demanda_por_establecimiento` (468) y `demanda_diaria` (13,152).

### Decisiones de diseño

**1. Corrección de la clave de semana en Plata, no en Bronce.**
Bronce conserva el dato original y auditable; la estandarización ocurre en la capa
responsable de ella. Impacto medido: 2,252 atenciones con clave incorrecta.

**2. Exclusión de semanas parciales en las tablas semanales.**
El dataset abarca del 01/01/2022 al 31/12/2024, por lo que `2021-W52` y `2025-W01`
tienen solo 2 días. Incluirlas produciría caídas artificiales del ~70% que
Isolation Forest marcaría como brotes falsos (S6). La serie analítica queda con
**156 semanas completas** (`2022-W01` a `2024-W52`).

**3. Fuente autoritativa de casos de dengue.**
Se usa `casos_dengue` (integrada desde MINSA/CDC vía JOIN) y no
`casos_dengue_semana` (desnormalizada en el HIS), porque la arquitectura define
MINSA/CDC como fuente de epidemiología. Ambas se conservan en Plata para
trazabilidad.

**4. Calibración de la oferta hospitalaria.**
La oferta se generaba con un rango fijo (1,500–2,200 consultas) sin relación con
la demanda real (~1,068 atenciones por establecimiento y semana), produciendo
brecha negativa en el 100% de los casos. Se recalibró contra la demanda observada.
Resultado: **brecha positiva en las 468 filas**, con un ratio demanda/oferta
mediano de **1.25**.

### Calidad de datos

- `atencion_id` único en las 500,000 filas — deduplicación correcta.
- Nulos máximos: **1.64%**, concentrados en las variables rezagadas por el inicio de la serie.
- Validación activa: el pipeline se detiene si aparecen claves de semana duplicadas.

### Pendientes identificados

- El dataset contiene 6 códigos CIE-10 distintos; el README (S1) plantea un análisis "top 20".
- `tasa_ocupacion` se sortea por fila sin relación con la demanda, por lo que el ranking entre establecimientos no tiene señal estadística.
- La tabla de brechas usa `demanda_atenciones − consultas_programadas` como proxy. El README define `OFERTA = Camas × Ocupación + Médicos × Productividad` y `BRECHA = DEMANDA predicha − OFERTA`. La fórmula completa se implementará en S6, cuando Prophet provea la demanda predicha.
- `consultas_ejecutadas` (mediana 876) es menor que `demanda_atenciones` (1,065) porque esta última incluye consulta externa, emergencia y hospitalización, mientras que las consultas programadas corresponden solo a consulta externa.